# ML Platform Research: Feature Engineering

## 1. Introduction

This notebook transforms the exploratory findings into reusable feature tables for the Gold layer.

The exploratory analysis identified several behavioral, temporal, session-based, and item-popularity signals that may be useful for predicting future purchases. The goal of this stage is to transform these signals into aggregated, leakage-aware features that can be generated consistently for each visitor and prediction date.

The feature engineering process follows a snapshot-based approach: for each visitor and date, features are calculated using only information available before that date. This prevents future events from influencing the predictors.

### 1.1 Prediction Horizon

The initial modeling objective considered predicting whether a visitor would make a purchase within the following 24 hours. However, the short prediction window produced a highly volatile target, making predictions more sensitive to short-term fluctuations and individual browsing patterns.

A 7-day prediction horizon was therefore selected for the final modeling task. Aggregating the outcome over a longer period reduces the variability of the target and provides a more stable representation of near-term purchase intent, while still preserving a sufficiently actionable prediction horizon.

Accordingly, this notebook generates a single target for the final modeling task:

`purchase_next_7d = 1` if the visitor makes at least one transaction from D through D+6.

## 2. Imports and Dataset Load

In [ ]:
import pandas as pd

import sys
sys.path.append("..")

from pathlib import Path

from utils import (
    create_sessions, 
    aggregate_session_statistics, 
    add_rolling_session_features, 
    create_purchase_target
)

events = pd.concat(
    [
        pd.read_parquet(file)
        for file in sorted(Path("../data/clean/events").glob("*.parquet"))
    ],
    ignore_index=True,
)

In [ ]:
events.head(5)

## 3. User Behavioral Features

The first group of features represents the visitor's historical interaction with the platform.

Event-level data is first aggregated by visitor and date. This intermediate representation allows rolling historical features to be calculated efficiently while maintaining the prediction-date structure required by the modeling stage.

### 3.1 Daily Event Aggregation

For each visitor and date, view, add-to-cart, and transaction events are converted into daily counts.

In [ ]:
events["date"] = events["timestamp"].dt.floor("D")

events["is_view"] = (events["event"] == "view").astype(int)
events["is_addtocart"] = (events["event"] == "addtocart").astype(int)
events["is_transaction"] = (events["event"] == "transaction").astype(int)

events

In [ ]:
events_daily_aggregation = (
    events
    .groupby(["visitorid", "date"])[["is_view", "is_addtocart", "is_transaction"]]
    .sum()
    .reset_index()
)

events_daily_aggregation = events_daily_aggregation.sort_values(
    ["visitorid", "date"]
)

events_daily_aggregation

### 3.2 Recent Interaction Features

In [ ]:
user_events_aggregation = (
    events_daily_aggregation
    .set_index("date")
    .groupby("visitorid")[["is_view", "is_addtocart"]]
    .rolling("7D", closed="left")
    .sum()
    .reset_index()
    .rename(columns={
        "is_view": "user_views_7d",
        "is_addtocart": "user_addtocart_7d"})
)

user_events_aggregation["user_views_7d"] = user_events_aggregation["user_views_7d"].fillna(0).astype("int32")
user_events_aggregation["user_addtocart_7d"] = user_events_aggregation["user_addtocart_7d"].fillna(0).astype("int16")

user_events_aggregation

In [ ]:
user_transactions_30d = (
    events_daily_aggregation
    .set_index("date")
    .groupby("visitorid")["is_transaction"]
    .rolling("30D", closed="left")
    .sum()
    .reset_index()
    .rename(columns={"is_transaction": "user_transactions_30d"})
)

user_transactions_30d["user_transactions_30d"] = user_transactions_30d["user_transactions_30d"].fillna(0).astype("int16")

user_transactions_30d

In [ ]:
user_aggregation = (
    user_events_aggregation
    .merge(
        right=user_transactions_30d,
        how="inner",
        on=["visitorid", "date"]
    )
)

user_aggregation

## 4. Session Features

Session-based features represent the intensity and duration of visitor activity.

A new session is defined when the time between consecutive events from the same visitor exceeds 30 minutes. Session-level statistics are then aggregated by visitor and date and transformed into rolling 30-day features.

### 4.1 Session Identification

> **Temporal consideration:** Sessions are assigned to the date of their start time. Therefore, sessions that cross midnight may contain events occurring after the assigned date. This behavior should be reviewed before using the session logic in a production pipeline.

In [ ]:
events = create_sessions(events)

events

### 4.2 Daily Session Statistics

After identifying sessions, session duration is calculated and aggregated by visitor and date.

In [ ]:
daily_session_stats = aggregate_session_statistics(events)

daily_session_stats

### 4.3 Rolling Session Features

The daily session statistics are transformed into historical 30-day features. The current prediction date is excluded from the rolling window to ensure that the features only use information available before the snapshot.

The resulting features represent:

- the number of sessions during the previous 30 days;
- the average session duration during the previous 30 days.

In [ ]:
session_features = add_rolling_session_features(
    daily_session_stats,
    user_aggregation
)

session_features

In [ ]:
user_aggregation = (
    user_aggregation
    .merge(
        right=session_features[["visitorid", "date", "user_sessions_30d", "avg_session_duration_30d"]],
        on=["visitorid", "date"],
        how='left'
    )
)

user_aggregation

## 5. Item Popularity Features

The exploratory analysis indicated that item purchases are highly concentrated among a relatively small subset of products. This suggests that item popularity may provide useful contextual information about a visitor's interactions.

Instead of using `itemid` directly as a model feature, historical item transaction volume is used to construct aggregate popularity signals.

All popularity measures are calculated using historical transactions only, excluding the current prediction date.

### 5.1 Daily Item Transaction History

First, transaction counts are aggregated by item and date.

A complete item-by-date grid is created so that days without transactions are explicitly represented as zero. This is necessary for correctly calculating rolling historical transaction counts.

In [ ]:
item_daily_transactions = (
    events.loc[
        events["event"] == "transaction",
        ["itemid", "date"]
    ]
    .groupby(["itemid", "date"])
    .size()
    .reset_index(name="transactions")
)


base = pd.MultiIndex.from_product(
    [
        events["itemid"].unique(),
        events["date"].unique()
    ],
    names=["itemid", "date"]
).to_frame(index=False)

item_daily_transactions = (
    base
    .merge(
        item_daily_transactions,
        on=["itemid", "date"],
        how="left"
    )
    .sort_values(["itemid", "date"])
)

item_daily_transactions["transactions"] = (
    item_daily_transactions["transactions"]
    .fillna(0)
    .astype("int16")
)

### 5.2 Historical Item Popularity

For each item and prediction date, the number of transactions during the previous 30 days is calculated. The current date is excluded from the rolling window to ensure that the feature only uses historical information.

In [ ]:
item_daily_transactions["item_transactions_30d"] = (
    item_daily_transactions
    .set_index("date")
    .groupby("itemid")["transactions"]
    .rolling("30D", closed="left")
    .sum()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

item_daily_transactions["item_transactions_30d"] = (
    item_daily_transactions["item_transactions_30d"]
    .fillna(0)
)

### 5.3 User Cart Item Popularity

The historical popularity of products added to the cart is aggregated at the visitor level.

For each visitor and prediction date, the average historical transaction volume of items added to the cart during the previous 7 days is calculated.

This feature combines recent purchase intent with the historical popularity of the products the visitor interacted with.

In [ ]:
cart_events = (
    events.loc[
        events["event"] == "addtocart",
        ["visitorid", "itemid", "date"]
    ]
    .copy()
)

In [ ]:
cart_events = cart_events.merge(
    item_daily_transactions[
        ["itemid", "date", "item_transactions_30d"]
    ],
    on=["itemid", "date"],
    how="left"
)

In [ ]:
daily_cart_item_stats = (
    cart_events
    .groupby(["visitorid", "date"])
    .agg(
        cart_item_popularity_sum=("item_transactions_30d", "sum"),
        cart_count=("itemid", "size")
    )
    .reset_index()
)

In [ ]:
daily_cart_item_stats = (
    daily_cart_item_stats
    .merge(
        user_aggregation[["visitorid", "date"]],
        on=["visitorid", "date"],
        how="right"
    )
    .fillna({
        "cart_item_popularity_sum": 0,
        "cart_count": 0
    })
)

In [ ]:
daily_cart_item_stats["cart_item_popularity_sum_7d"] = (
    daily_cart_item_stats
    .set_index("date")
    .groupby("visitorid")["cart_item_popularity_sum"]
    .rolling("7D", closed="left")
    .sum()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

daily_cart_item_stats["cart_count_7d"] = (
    daily_cart_item_stats
    .set_index("date")
    .groupby("visitorid")["cart_count"]
    .rolling("7D", closed="left")
    .sum()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

In [ ]:
daily_cart_item_stats["user_cart_items_avg_popularity_7d"] = (
    daily_cart_item_stats["cart_item_popularity_sum_7d"]
    / daily_cart_item_stats["cart_count_7d"]
).fillna(0)

In [ ]:
user_aggregation = user_aggregation.merge(
    daily_cart_item_stats[
        [
            "visitorid",
            "date",
            "user_cart_items_avg_popularity_7d"
        ]
    ],
    on=["visitorid", "date"],
    how="left"
)

user_aggregation["user_cart_items_avg_popularity_7d"] = (
    user_aggregation["user_cart_items_avg_popularity_7d"]
    .fillna(0)
)

user_aggregation

### 5.4 Top-Seller Cart Ratio

A second item-popularity signal measures how frequently the visitor adds highly popular products to the cart.

For each prediction date, items are classified as top sellers when their historical transaction volume is in the top 10% of items. The feature then measures the proportion of cart interactions during the previous 7 days involving these highly popular items.

In [ ]:
top_10_threshold = (
    item_daily_transactions
    .groupby("date")["item_transactions_30d"]
    .quantile(0.90)
    .rename("top_10_threshold")
    .reset_index()
)

item_daily_transactions = item_daily_transactions.merge(
    top_10_threshold,
    on="date",
    how="left"
)

item_daily_transactions["is_top_seller"] = (
    item_daily_transactions["item_transactions_30d"]
    >= item_daily_transactions["top_10_threshold"]
)

In [ ]:
cart_events = cart_events.merge(
    item_daily_transactions[
        ["itemid", "date", "is_top_seller"]
    ],
    on=["itemid", "date"],
    how="left"
)

In [ ]:
daily_cart_stats = (
    cart_events
    .groupby(["visitorid", "date"])
    .agg(
        cart_count=("itemid", "size"),
        top_seller_cart_count=("is_top_seller", "sum")
    )
    .reset_index()
)

In [ ]:
daily_cart_stats = (
    daily_cart_stats
    .merge(
        user_aggregation[["visitorid", "date"]],
        on=["visitorid", "date"],
        how="right"
    )
    .fillna({
        "cart_count": 0,
        "top_seller_cart_count": 0
    })
)

In [ ]:
daily_cart_stats["cart_count_7d"] = (
    daily_cart_stats
    .set_index("date")
    .groupby("visitorid")["cart_count"]
    .rolling("7D", closed="left")
    .sum()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

daily_cart_stats["top_seller_cart_count_7d"] = (
    daily_cart_stats
    .set_index("date")
    .groupby("visitorid")["top_seller_cart_count"]
    .rolling("7D", closed="left")
    .sum()
    .reset_index(level=0, drop=True)
    .to_numpy()
)

In [ ]:
daily_cart_stats["user_cart_top_seller_ratio_7d"] = (
    daily_cart_stats["top_seller_cart_count_7d"]
    / daily_cart_stats["cart_count_7d"]
).fillna(0)

In [ ]:
user_aggregation = user_aggregation.merge(
    daily_cart_stats[
        [
            "visitorid",
            "date",
            "user_cart_top_seller_ratio_7d"
        ]
    ],
    on=["visitorid", "date"],
    how="left"
)

user_aggregation["user_cart_top_seller_ratio_7d"] = (
    user_aggregation["user_cart_top_seller_ratio_7d"]
    .fillna(0)
)

user_aggregation

## 6. Purchase Target

The target represents whether a visitor makes at least one purchase within the selected 7-day prediction window.

### 6.1 Target Definition

For a prediction date `D`, the target is equal to 1 if the visitor makes a transaction on any day from `D` through `D+6`.

The prediction date itself is included in the target definition because the prediction window is defined relative to the snapshot date.

### 6.2 Target Construction and Dataset Saving

In [ ]:
daily_purchases = (
    events.loc[
        events["event"] == "transaction",
        ["visitorid", "date"]
    ]
    .drop_duplicates()
)

In [ ]:
user_aggregation_target = create_purchase_target(
        user_aggregation,
        daily_purchases,
        7
    )

user_aggregation_target

In [ ]:
user_aggregation_target.to_parquet("../data/processed/processed_data.parquet", index=False)